# SST and SST difference from WOA23

Related issue: https://github.com/ACCESS-Community-Hub/access-om3-25km-paper-1/issues/9

In [ ]:
# These first two cells must be in all notebooks!
# It allows us to run all the notebooks at once, this cell has a tag "parameters" which allows us to pass in 
# arguments externally using papermill (see mkfigs.sh for details)

# Set esm_file to the datastore for the main experiment of interest

# Second experiment to compare against (set to None to skip the comparison)
esm_file_compare = None
esm_file = "/g/data/ol01/outputs/access-om3-25km/MC_25km_jra_iaf+wombatlite-test3v2-00532b88/datastore.json"
# esm_file = "/g/data/zv30/non-cmip/ACCESS-CM3/cm3-run-27-07-2026-PD-control/cm3-datastore/cm3-datastore.json"

# What physical field you want, in CF terms (stays the same across models)
variable_standard_name = "sea_surface_temperature"

# Fallback variable name to use if the catalog doesn’t expose standard_name
fallback_variable_names = ["tos", "sst", "thetao", "temp"]

# Reference / observational dataset (we'll use this later when we get to the WOA block)
obs_file_pattern = "/g/data/ik11/inputs/access-om3/woa23/025/2025.08.26/woa23_ts_*"  # matches 25km/0.25deg grid
# obs_file_pattern = "/g/data/ik11/inputs/access-om3/woa23/1deg/2026.02.17/woa23_ts_*"  # matches the 100km/1deg grid
obs_var_name = "temp"

# Frequency depends on the datastore you’re using:
data_frequency = "1mon"

# papermill settings. *No need to modify these if running interactively.* 
papermill = False                      # `cwd` and `nbname` will be populated by papermill.
cwd = None                             # current working directory 
nbname = None                          # notebook name

In [ ]:
if not papermill: 
    import nci_ipynb, os  # requires conda/analysis3-26.03 or later
    cwd = nci_ipynb.dir()
    nbname = nci_ipynb.name()
    os.chdir(cwd)
import mkfigs_bootstrap  # noqa: adds external/access-model-mkfigs/src to sys.path (stop-gap)
from mkfigs import MkmdWriter
mkmd = MkmdWriter(esm_file, nbname, str(cwd), pm=papermill)
from exptdata_access import get_experiment_info, guess_experiment_from_esm_file
from model_agnostic import get_lon_lat_from_catalog, select_variable, patch_broken_conda_env
patch_broken_conda_env()  # conda env workaround for analysis3-26.03+; no-op on healthy envs
# Infer model information from the datastore path
expt_key, info = guess_experiment_from_esm_file(esm_file)
model_name = info["model"]             # OM3 or CM3

In [ ]:
import xarray as xr
import cf_xarray as cfxr
import cf_xarray.units
import pint_xarray
from pint import application_registry as ureg
import intake
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
from distributed import Client
import cftime
import os
import cmocean as cm
import cartopy.feature as cft
from textwrap import wrap
xr.set_options(keep_attrs=True);  # cf_xarray works best when xarray keeps attributes by default

In [ ]:
from model_agnostic import patch_dask_workers
client = Client(threads_per_worker=1)
patch_dask_workers(client)  # patch workers too
client

### Define plot function

In [ ]:
blue_marble = plt.imread('/g/data/ik11/grids/BlueMarble.tiff')
blue_marble_extent = (-180, 180, -90, 90)

In [ ]:
def plot(dat, title=None, projection=None, add_blue_marble=True, **kwargs):
    """
    Generic 2D map plot helper for model / obs / bias fields.

    - Works for any variable (SST, SSH, salinity, etc.).
    - Assumes the last two dimensions are (y, x) horizontal dims.
    - Uses contourf with a cartopy projection.
    """

    import matplotlib.pyplot as plt
    import cartopy.crs as ccrs
    from textwrap import wrap

    # Make sure we're working with a DataArray
    da = dat
    if not isinstance(da, xr.DataArray):
        raise TypeError("plot() expects an xarray.DataArray")

    # Handle simple time dimension cases (e.g. time-mean or single time step)
    if da.ndim == 3 and "time" in da.dims and da.sizes["time"] == 1:
        da = da.isel(time=0, drop=True)

    if da.ndim != 2:
        raise ValueError(
            f"plot() expects a 2D field or a 3D field with singleton time; got dims {da.dims}"
        )

    # Infer horizontal dims as the last two dims
    ydim, xdim = da.dims[-2], da.dims[-1]

    # Title and colourbar label from attributes
    long_name = da.attrs.get("long_name", da.name or "Field")
    units = da.attrs.get("units", "")

    if title is None:
        title = long_name

    cbar_label = long_name if units == "" else f"{long_name} [{units}]"
    cbar_label = "\n".join(wrap(cbar_label, 45))

    # Projection: default to Robinson
    if projection is None:
        projection = ccrs.Robinson(central_longitude=-100)

    fig = plt.figure(figsize=(12, 6))
    ax = plt.axes(projection=projection)

    # Main filled contour plot
    da.plot.contourf(
        ax=ax,
        x=xdim,
        y=ydim,
        transform=ccrs.PlateCarree(),
        cbar_kwargs={
            "label": cbar_label,
            "fraction": 0.03,
            "aspect": 15,
            "shrink": 0.7,
        },
        **kwargs,
    )

    # Optional blue marble overlay if available
    if add_blue_marble and "blue_marble" in globals() and "blue_marble_extent" in globals():
        ax.imshow(
            blue_marble,
            extent=blue_marble_extent,
            transform=ccrs.PlateCarree(),
            origin="upper",
        )

    ax.coastlines()
    plt.title(title)
    plt.tight_layout()

### Load and plot data from ESM datastore

In [ ]:
from exptdata_access import get_exptname_from_path
exptname=get_exptname_from_path(esm_file)
print("Experiment name:", exptname)

datastore = intake.open_esm_datastore(
    esm_file,
    columns_with_iterables=[
        "variable",
        "variable_long_name",
        "variable_standard_name",
        "variable_cell_methods",
        "variable_units"
    ]
)

In [ ]:
da_model = select_variable(
    datastore,
    variable_standard_name,
    fallback_variable_names,
    data_frequency=data_frequency,
)
print("Selected variable:", da_model.name)
print("dims:", da_model.dims)

In [ ]:
# Load model grid and attach true lon/lat to the model field
# (uses the catalog-driven helper, robust to both ACCESS-OM3's output000/*.mom6.static.nc
# layout and ACCESS-CM3's differently-named/located static files -- a hardcoded
# "output000/access-om3.mom6.static.nc" path only exists for ACCESS-OM3)
geolon, geolat = get_lon_lat_from_catalog(datastore)

model_all = da_model.cf.assign_coords(
    {
        "longitude": geolon,
        "latitude": geolat,
    }
)
print("Attached geolon/geolat from catalog grid variables.")
print("model_all dims:", model_all.dims)

In [ ]:
averaging_mode = "last_n_years"   # or "full_period" / "fixed_period"
averaging_last_n_years = 10
averaging_start_date = None
averaging_end_date   = None

In [ ]:
# Put model time axis on a common calendar
model_all = model_all.convert_calendar("proleptic_gregorian", use_cftime=True)

# Inspect full time coverage
t0 = model_all.time.values[0]
t1 = model_all.time.values[-1]
print("Full model time range:", t0, "→", t1)

# Decide the averaging window based on configuration
if averaging_mode == "full_period":
    datestart = t0
    datestop = t1

elif averaging_mode == "last_n_years":
    datestop = t1
    # Subtract N years in a calendar-aware way
    datelist = list(cftime.to_tuple(datestop))
    datelist[0] -= averaging_last_n_years
    datestart = cftime.datetime(*datelist, calendar=datestop.calendar)

elif averaging_mode == "fixed_period":
    if averaging_start_date is None or averaging_end_date is None:
        raise ValueError(
            "averaging_mode='fixed_period' requires averaging_start_date and averaging_end_date"
        )
    # Use cftime to create start/end on the same calendar
    datestart = xr.cftime_range(
        start=averaging_start_date,
        periods=1,
        calendar="proleptic_gregorian",
    )[0]
    datestop = xr.cftime_range(
        start=averaging_end_date,
        periods=1,
        calendar="proleptic_gregorian",
    )[0]
else:
    raise ValueError(f"Unknown averaging_mode: {averaging_mode!r}")

timerange = slice(datestart, datestop)
print("Averaging window:", timerange)

# Restrict model to this configured window
model_window = model_all.cf.sel(time=timerange)
print("Windowed dims:", model_window.dims)

In [ ]:
%%time
model = (
    model_window
    .cf.mean("time")          # CF-aware: whatever the time dim is called
    .pint.quantify()          # attach Pint units from attrs["units"]
    .pint.to("degC")          # convert to degrees Celsius
    .pint.dequantify()        # back to plain DataArray for plotting / saving
)

print("Final model dims:", model.dims)
print("Final model units:", model.attrs.get("units"))

In [ ]:
long_name = model.attrs.get("long_name", "SST")

plot(
    model,
    levels=39,
    vmin=-3,
    vmax=35,
    extend="both",
    cmap=cm.cm.thermal,
    title=(
        f"{long_name} "
        f"{datestart.strftime('%Y-%m-%d')} – {datestop.strftime('%Y-%m-%d')} "
        f"mean\nin {exptname}"
    ),
)
mkmd.savefig(plt.gcf(), "Sea Surface Temperature", "ACCESS-OM3 sea surface temperature. [GitHub issue: Global SST bias](https://github.com/ACCESS-Community-Hub/access-om3-paper-1/issues/9)")

### Load and plot data from MC comparison experiment

In [ ]:
if esm_file_compare is not None:
    exptname_compare = get_exptname_from_path(esm_file_compare)
    print("Comparison experiment name:", exptname_compare)

    datastore_compare = intake.open_esm_datastore(
        esm_file_compare,
        columns_with_iterables=[
            "variable",
            "variable_long_name",
            "variable_standard_name",
            "variable_cell_methods",
            "variable_units"
        ]
    )

    da_model_compare = select_variable(
        datastore_compare,
        variable_standard_name,
        fallback_variable_names,
        data_frequency=data_frequency,
    )
    print("Selected variable:", da_model_compare.name)
    print("dims:", da_model_compare.dims)

In [ ]:
if esm_file_compare is not None:
    geolon_compare, geolat_compare = get_lon_lat_from_catalog(datastore_compare)

    model_all_compare = da_model_compare.cf.assign_coords(
        {
            "longitude": geolon_compare,
            "latitude": geolat_compare,
        }
    )
    print("model_all_compare dims:", model_all_compare.dims)

In [ ]:
if esm_file_compare is not None:
    # Put comparison model time axis on a common calendar
    model_all_compare = model_all_compare.convert_calendar("proleptic_gregorian", use_cftime=True)

    # Computed independently from the main experiment's own window (not a
    # shared min() across both) -- the two runs' record lengths/calendars
    # need not line up.
    t0_compare = model_all_compare.time.values[0]
    t1_compare = model_all_compare.time.values[-1]
    print("Full comparison model time range:", t0_compare, "→", t1_compare)

    # Use the SAME averaging window as the main experiment, so differences between
    # the two are not confounded by them covering different periods. Clamp to the
    # comparison run's own coverage if it does not span the main window.
    datestart_compare = max(datestart, t0_compare)
    datestop_compare = min(datestop, t1_compare)

    if datestop_compare <= datestart_compare:
        raise ValueError(
            f"{exptname_compare} covers {t0_compare} - {t1_compare}, which does not overlap "
            f"the main averaging window {datestart} - {datestop}"
        )
    if (datestart_compare != datestart) or (datestop_compare != datestop):
        print(
            f"WARNING: {exptname_compare} does not span the main averaging window "
            f"{datestart.strftime('%Y-%m-%d')} - {datestop.strftime('%Y-%m-%d')}; clamped to "
            f"{datestart_compare.strftime('%Y-%m-%d')} - {datestop_compare.strftime('%Y-%m-%d')}"
        )

    timerange_compare = slice(datestart_compare, datestop_compare)
    print("Comparison averaging window:", timerange_compare)

    model_window_compare = model_all_compare.cf.sel(time=timerange_compare)
    print("Windowed dims:", model_window_compare.dims)

In [ ]:
if esm_file_compare is not None:
    model_compare = (
        model_window_compare
        .cf.mean("time")
        .pint.quantify()
        .pint.to("degC")
        .pint.dequantify()
    )
    print("Final model_compare units:", model_compare.attrs.get("units"))

In [ ]:
if esm_file_compare is not None:
    long_name_compare = model_compare.attrs.get("long_name", "SST")

    plot(
        model_compare,
        levels=39,
        vmin=-3,
        vmax=35,
        extend="both",
        cmap=cm.cm.thermal,
        title=(
            f"{long_name_compare} "
            f"{datestart_compare.strftime('%Y-%m-%d')} – {datestop_compare.strftime('%Y-%m-%d')} "
            f"mean\nin {exptname_compare}"
        ),
    )
    mkmd.savefig(plt.gcf(), "Sea Surface Temperature (comparison)", f"{exptname_compare} sea surface temperature. [GitHub issue: Global SST bias](https://github.com/ACCESS-Community-Hub/access-om3-paper-1/issues/9)")

### Load data from WOA23 (annual mean; Jan is ACCESS-OM3 initial condition)

In [ ]:
# Load observational / reference data (e.g. WOA23) in a model-agnostic way.
# WOA23 products by model grid shape: the obs is assigned directly onto the model
# grid below, so the two must have identical shape. Pick the product that matches
# rather than hard-coding one resolution (lets this notebook run at 25km and 100km).
woa_products = {
    (1152, 1440): "/g/data/ik11/inputs/access-om3/woa23/025/2025.08.26/woa23_ts_*",   # 25km / 0.25deg grid
    (324, 360):   "/g/data/ik11/inputs/access-om3/woa23/1deg/2026.02.17/woa23_ts_*",  # 100km / 1deg grid
}

def load_woa_on_grid(lon2d, lat2d, var_name=None, default_pattern=None):
    """Load the WOA23 product matching this model grid, and place it on that grid.

    Returns (obs_on_native_dims, obs_with_model_lon_lat_coords).
    """
    var_name = obs_var_name if var_name is None else var_name
    default_pattern = obs_file_pattern if default_pattern is None else default_pattern
    shape = tuple(lat2d.shape[-2:])
    pattern = woa_products.get(shape, default_pattern)
    print(f"model grid {shape} -> obs {pattern}")
    ds_obs = xr.open_mfdataset(pattern, chunks={"time": -1})
    da_obs = ds_obs[var_name]
    try:
        da_obs_sfc = da_obs.cf.isel(Z=0)          # first depth level
    except Exception:
        da_obs_sfc = da_obs.isel(depth=0)         # fallback if depth is literally 'depth'
    o = da_obs_sfc.cf.mean("time")
    if {"yh", "xh"}.issubset(o.dims):
        grid = o
    elif {"lat", "lon"}.issubset(o.dims):
        grid = o.rename({"lat": "yh", "lon": "xh"})
    else:
        raise ValueError("Obs grid dims not recognized; expected yh/xh or lat/lon")
    return grid, grid.cf.assign_coords({"longitude": lon2d, "latitude": lat2d})

obs_grid, obs = load_woa_on_grid(geolon, geolat)

print("Obs dims:", obs.dims)
print("Obs coords:", list(obs.coords))

### Plot model minus WOA23
First plot full range, then a sequence at specified ranges.

**BUG? is WOA initial condition data conservative temperature, but model data potential temperature? - CHECK!**

In [ ]:
# 1) Obs is already on the model grid with geolon/geolat coordinates
obs_on_model = obs

# 2) Compute bias directly (units already consistent)
bias = (model - obs_on_model).load()
bias.attrs = model.attrs

# 3) Plot the bias
long_name = model.attrs.get("long_name", "SST")

plot(
    bias,
    levels=61,
    extend="both",
    cmap="seismic",
    title=(
        f"{long_name} bias "
        f"{datestart.strftime('%Y-%m-%d')} – {datestop.strftime('%Y-%m-%d')} "
        f"mean\n{exptname} minus WOA23"
    ),
)

In [ ]:
plot(
    bias,
    levels=61,
    vmin=-3,
    vmax=3,
    extend="both",
    cmap="seismic",
    title=(
        f"{long_name} bias"
        f"{datestart.strftime('%Y-%m-%d')} – {datestop.strftime('%Y-%m-%d')} "
        f"mean\n{exptname} minus WOA23"
    ),
)
mkmd.savefig(plt.gcf(), "Sea Surface Temperature Bias", "ACCESS-OM3 sea surface temperature minus WOA2023. [GitHub issue: Global SST bias](https://github.com/ACCESS-Community-Hub/access-om3-paper-1/issues/9)")

In [ ]:
if esm_file_compare is not None:
    # Obs on the comparison model's own grid. If it is a different resolution to
    # the main model, re-load the matching WOA product rather than reusing obs_grid.
    if tuple(geolat_compare.shape[-2:]) == tuple(geolat.shape[-2:]):
        obs_compare_grid = obs_grid.cf.assign_coords(
            {
                "longitude": geolon_compare,
                "latitude": geolat_compare,
            }
        )
    else:
        _, obs_compare_grid = load_woa_on_grid(geolon_compare, geolat_compare)

In [ ]:
if esm_file_compare is not None:
    bias_compare = (model_compare - obs_compare_grid).load()
    bias_compare.attrs = model_compare.attrs

    plot(
        bias_compare,
        levels=61,
        vmin=-3,
        vmax=3,
        extend="both",
        cmap="seismic",
        title=(
            f"{long_name_compare} bias "
            f"{datestart_compare.strftime('%Y-%m-%d')} – {datestop_compare.strftime('%Y-%m-%d')} "
            f"mean\n{exptname_compare} minus WOA23"
        ),
    )
    mkmd.savefig(plt.gcf(), "Sea Surface Temperature Bias (comparison)", f"{exptname_compare} sea surface temperature minus WOA2023. [GitHub issue: Global SST bias](https://github.com/ACCESS-Community-Hub/access-om3-paper-1/issues/9)")

In [ ]:
client.close()